In [ ]:
#Cau 1
import cv2
import numpy as np
import random

def process_image(image_path="a.jpg"):
    """
    Performs various image processing operations on the given image.

    Args:
        image_path (str): The path to the input image file.
    """

    # Load the image
    img = cv2.imread(image_path)

    if img is None:
        print(f"Error: Could not load image from {image_path}. Please make sure the file exists.")
        return

    # --- Mean Filter (0.5 Points) ---
    print("Applying Mean Filter...")
    # Define kernel size for the mean filter (e.g., 5x5)
    kernel_size = (5, 5)
    mean_filtered_img = cv2.blur(img, kernel_size)
    cv2.imwrite("a_mean_filtered.jpg", mean_filtered_img)
    print("Mean filtered image saved as a_mean_filtered.jpg")

    # --- Edge Detection Filter (0.5 Points) ---
    print("Applying Edge Detection Filter (Sobel)...")
    # Convert to grayscale for edge detection
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply Sobel filter in X and Y directions
    sobelx = cv2.Sobel(gray_img, cv2.CV_64F, 1, 0, ksize=5)
    sobely = cv2.Sobel(gray_img, cv2.CV_64F, 0, 1, ksize=5)

    # Combine X and Y gradients
    edge_detected_img = cv2.magnitude(sobelx, sobely)
    # Normalize to 0-255 for saving
    edge_detected_img = cv2.normalize(edge_detected_img, None, 0, 255, cv2.NORM_MINMAX)
    edge_detected_img = np.uint8(edge_detected_img)

    cv2.imwrite("a_edge_detected.jpg", edge_detected_img)
    print("Edge detected image saved as a_edge_detected.jpg")

    # --- Random BGR to RGB Color Change (0.5 Points) ---
    print("Changing image to a random RGB color space...")
    # Split the BGR channels
    b, g, r = cv2.split(img)

    # Randomly reorder or scale channels. For simplicity and to ensure it's still
    # a visually distinct image, let's randomly scale each channel.
    # We can also randomly swap them or assign random values, but scaling keeps
    # some semblance of the original image's structure.
    
    # Generate random scaling factors between 0.5 and 1.5 for each channel
    scale_b = random.uniform(0.5, 1.5)
    scale_g = random.uniform(0.5, 1.5)
    scale_r = random.uniform(0.5, 1.5)

    # Apply scaling and clip values to 0-255
    b_new = np.clip(b * scale_b, 0, 255).astype(np.uint8)
    g_new = np.clip(g * scale_g, 0, 255).astype(np.uint8)
    r_new = np.clip(r * scale_r, 0, 255).astype(np.uint8)
    
    # Merge the modified channels back (in RGB order for a random RGB feel)
    # Note: OpenCV's default is BGR, so if we want to explicitly save as an RGB-like
    # output where the channels are perceived as R, G, B, we merge them in that order.
    # However, saving with cv2.imwrite will still interpret it as BGR unless specified
    # differently or if we convert it back to BGR from this "conceptual" RGB.
    # For a *truly* random color change, let's just randomly reassign the original
    # B, G, R channels to new positions or modify them more drastically.
    
    # Let's perform a random permutation of the channels and then combine them.
    channels = [b, g, r]
    random.shuffle(channels) # Randomly shuffles the original B, G, R channels
    random_color_img = cv2.merge(channels)

    cv2.imwrite("a_random_color.jpg", random_color_img)
    print("Random colored image saved as a_random_color.jpg")

    # --- BGR to HSV Conversion and Channel Separation (0.5 Points) ---
    print("Converting to HSV and separating channels...")
    hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Split HSV channels
    h, s, v = cv2.split(hsv_img)

    # Save each channel as a grayscale image
    cv2.imwrite("a_hue.jpg", h)
    cv2.imwrite("a_saturation.jpg", s)
    cv2.imwrite("a_value.jpg", v)
    print("Hue, Saturation, and Value channels saved as a_hue.jpg, a_saturation.jpg, a_value.jpg")

# Example usage:
# Make sure you have an image named 'a.jpg' in the same directory as this script.
process_image("a.jpg")

Applying Mean Filter...
Mean filtered image saved as a_mean_filtered.jpg
Applying Edge Detection Filter (Sobel)...
Edge detected image saved as a_edge_detected.jpg
Changing image to a random RGB color space...
Random colored image saved as a_random_color.jpg
Converting to HSV and separating channels...
Hue, Saturation, and Value channels saved as a_hue.jpg, a_saturation.jpg, a_value.jpg


In [ ]:
#Cau 2
import cv2
import numpy as np
import random
import os

# --- 1. Các Hàm Biến đổi Ảnh ---

def image_inverse_transformation(image):
    """Áp dụng phép biến đổi nghịch đảo (âm bản) cho ảnh."""
    print("   -> Đang áp dụng biến đổi nghịch đảo...")
    return 255 - image

def gamma_correction(image):
    """Áp dụng hiệu chỉnh Gamma với giá trị gamma ngẫu nhiên từ 0.5 đến 2.0."""
    gamma = random.uniform(0.5, 2.0)
    print(f"   -> Đang áp dụng hiệu chỉnh Gamma với gamma = {gamma:.2f}...")
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(256)]).astype("uint8")
    return cv2.LUT(image, table)

def log_transformation(image):
    """Áp dụng biến đổi Log với hệ số nhân ngẫu nhiên từ 1.0 đến 5.0."""
    c = random.uniform(1.0, 5.0)
    print(f"   -> Đang áp dụng biến đổi Log với hệ số c = {c:.2f}...")
    log_transformed_image = c * np.log1p(image.astype(np.float32))
    return cv2.normalize(log_transformed_image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def histogram_equalization(image):
    """
    Áp dụng cân bằng lược đồ mức xám (Histogram Equalization).
    Đối với ảnh màu, nó được áp dụng cho kênh Y (độ sáng) trong không gian màu YUV.
    """
    print("   -> Đang áp dụng cân bằng lược đồ mức xám...")
    if len(image.shape) == 3: # Ảnh màu
        img_yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
        img_yuv[:,:,0] = cv2.equalizeHist(img_yuv[:,:,0])
        return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)
    else: # Ảnh mức xám
        return cv2.equalizeHist(image)

def contrast_stretching(image):
    """Áp dụng kéo giãn độ tương phản với giá trị min và max ngẫu nhiên từ 0 đến 255."""
    r_min = random.randint(0, 100)
    r_max = random.randint(155, 255) 
    
    if r_min >= r_max:
        r_min, r_max = r_max, r_min if r_max < r_min else (r_min, min(255, r_min + 50))
        if r_min > r_max: 
            r_min = 0 
            r_max = 255

    print(f"   -> Đang áp dụng kéo giãn độ tương phản với min={r_min}, max={r_max}...")
    
    table = np.zeros((256,), dtype=np.uint8)
    for i in range(256):
        if i < r_min:
            table[i] = 0
        elif i > r_max:
            table[i] = 255
        else:
            table[i] = int(255 * ((i - r_min) / (r_max - r_min)))
    
    return cv2.LUT(image, table)

def adaptive_histogram_equalization(image):
    """
    Áp dụng cân bằng lược đồ mức xám thích ứng (CLAHE) với ô lưới 8x8.
    Đối với ảnh màu, nó được áp dụng cho kênh Giá trị (V) trong không gian màu HSV.
    """
    print("   -> Đang áp dụng cân bằng lược đồ mức xám thích ứng (CLAHE)...")
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    
    if len(image.shape) == 3: # Ảnh màu
        hsv_img = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv_img)
        v = clahe.apply(v)
        return cv2.cvtColor(cv2.merge([h, s, v]), cv2.COLOR_HSV2BGR)
    else: # Ảnh mức xám
        return clahe.apply(image)

# --- 2. Hàm Hiển thị Menu và Xử lý Chính ---

def display_menu():
    """Hiển thị menu các phương pháp biến đổi ảnh và hướng dẫn sử dụng phím tắt."""
    print("\n" + "="*50)
    print("       MENU BIẾN ĐỔI ẢNH (NHẤN PHÍM ĐỂ THỰC HIỆN)")
    print("="*50)
    print("  [I] : Biến đổi ảnh nghịch đảo (Inverse Transformation)")
    print("  [G] : Hiệu chỉnh Gamma (Gamma-Correction)")
    print("  [L] : Biến đổi Log (Log Transformation)")
    print("  [H] : Cân bằng lược đồ mức xám (Histogram Equalization)")
    print("  [C] : Kéo giãn độ tương phản (Contrast Stretching)")
    print("  [A] : Cân bằng lược đồ mức xám thích ứng (Adaptive Histogram Equalization - CLAHE)")
    print("  [Q] : Thoát chương trình")
    print("="*50)
    print("  *** HƯỚNG DẪN: Nhấp chuột vào MỘT TRONG CÁC CỬA SỔ ẢNH (Ảnh Gốc 1, 2, 3...)")
    print("                 sau đó nhấn phím tương ứng trên bàn phím để áp dụng biến đổi.")
    print("                 Các cửa sổ ảnh biến đổi sẽ hiện ra.")
    print("                 Nhấn 'Q' để đóng tất cả và thoát.")
    print("="*50)

def main():
    """Hàm chính điều khiển chương trình biến đổi ảnh."""
    image_names = ["image1.jpg", "image2.jpg", "image3.jpg"]
    loaded_images = []
    
    print("--- KHỞI ĐỘNG CHƯƠNG TRÌNH XỬ LÝ ẢNH ---")
    for i, name in enumerate(image_names):
        if not os.path.exists(name):
            print(f"LỖI NGHIÊM TRỌNG: Không tìm thấy ảnh '{name}'.")
            print("Vui lòng đảm bảo các ảnh image1.jpg, image2.jpg, image3.jpg")
            print("được đặt CÙNG THƯ MỤC với file script này.")
            print("Chương trình sẽ thoát.")
            return # Thoát nếu không tìm thấy ảnh cần thiết
        
        img = cv2.imread(name)
        if img is None:
            print(f"LỖI: Không thể tải ảnh '{name}'. Tệp có thể bị hỏng hoặc không phải định dạng ảnh hợp lệ.")
            print("Chương trình sẽ thoát.")
            return
        
        loaded_images.append((img, name, i + 1)) # Lưu ảnh, tên gốc và số thứ tự
        cv2.imshow(f"Ảnh Gốc {i+1}: {name}", img)
        print(f"  -> Đã tải thành công ảnh: '{name}'")
    
    print("\nĐã tải tất cả ảnh gốc. Bây giờ, hãy xem menu.")
    display_menu()

    while True:
        # cv2.waitKey(0) sẽ đợi vô thời hạn cho một phím nhấn
        # Đảm bảo cửa sổ OpenCV đang được focus (nhấp chuột vào cửa sổ ảnh)
        key = cv2.waitKey(0) & 0xFF 
        
        # Thoát khi nhấn 'Q' hoặc 'q'
        if key == ord('q') or key == ord('Q'):
            print("Bạn đã chọn thoát. Đang đóng tất cả cửa sổ và kết thúc chương trình.")
            break

        transformation_applied = False
        method_name = ""
        transform_func = None

        if key == ord('i') or key == ord('I'):
            print("\nBạn đã chọn: Biến đổi ảnh nghịch đảo (Phím I)")
            transform_func = image_inverse_transformation
            method_name = "inverse"
            transformation_applied = True
        elif key == ord('g') or key == ord('G'):
            print("\nBạn đã chọn: Hiệu chỉnh Gamma (Phím G)")
            transform_func = gamma_correction
            method_name = "gamma"
            transformation_applied = True
        elif key == ord('l') or key == ord('L'):
            print("\nBạn đã chọn: Biến đổi Log (Phím L)")
            transform_func = log_transformation
            method_name = "log"
            transformation_applied = True
        elif key == ord('h') or key == ord('H'):
            print("\nBạn đã chọn: Cân bằng lược đồ mức xám (Phím H)")
            transform_func = histogram_equalization
            method_name = "histogram"
            transformation_applied = True
        elif key == ord('c') or key == ord('C'):
            print("\nBạn đã chọn: Kéo giãn độ tương phản (Phím C)")
            transform_func = contrast_stretching
            method_name = "contrast"
            transformation_applied = True
        elif key == ord('a') or key == ord('A'):
            print("\nBạn đã chọn: Cân bằng lược đồ mức xám thích ứng (Phím A)")
            transform_func = adaptive_histogram_equalization
            method_name = "clahe"
            transformation_applied = True
        else:
            print(f"Phím '{chr(key)}' không hợp lệ. Vui lòng nhấn một phím từ menu (I, G, L, H, C, A hoặc Q).")
            display_menu() # Hiển thị lại menu
            continue # Tiếp tục vòng lặp để đợi phím hợp lệ

        if transformation_applied:
            print(f"Bắt đầu áp dụng biến đổi '{method_name}' cho tất cả các ảnh...")
            for original_img, original_name, img_num in loaded_images:
                # Tạo một bản sao của ảnh gốc để biến đổi, tránh thay đổi ảnh gốc ban đầu
                transformed_img = transform_func(original_img.copy())
                
                # Lưu kết quả vào file với tên định dạng output_[phương pháp]_[số ảnh].jpg
                output_filename = f"output_{method_name}_{img_num}.jpg"
                cv2.imwrite(output_filename, transformed_img)
                print(f"  -> Đã lưu kết quả của ảnh gốc '{original_name}' vào: '{output_filename}'")
                
                # Hiển thị ảnh đã biến đổi trong một cửa sổ riêng
                # Tên cửa sổ cần phải duy nhất để tránh ghi đè
                cv2.imshow(f"Ảnh {img_num} - {method_name.replace('_', ' ').title()}", transformed_img)
            
            print("\nHoàn tất áp dụng biến đổi cho tất cả các ảnh.")
            print("Chương trình đang đợi bạn nhấn phím biến đổi khác, hoặc 'Q' để thoát.")
            display_menu() # Hiển thị lại menu sau khi hoàn tất biến đổi

    cv2.destroyAllWindows() # Đóng tất cả cửa sổ khi thoát chương trình

if __name__ == "__main__":
    main()

--- KHỞI ĐỘNG CHƯƠNG TRÌNH XỬ LÝ ẢNH ---
  -> Đã tải thành công ảnh: 'image1.jpg'
  -> Đã tải thành công ảnh: 'image2.jpg'
  -> Đã tải thành công ảnh: 'image3.jpg'

Đã tải tất cả ảnh gốc. Bây giờ, hãy xem menu.

       MENU BIẾN ĐỔI ẢNH (NHẤN PHÍM ĐỂ THỰC HIỆN)
  [I] : Biến đổi ảnh nghịch đảo (Inverse Transformation)
  [G] : Hiệu chỉnh Gamma (Gamma-Correction)
  [L] : Biến đổi Log (Log Transformation)
  [H] : Cân bằng lược đồ mức xám (Histogram Equalization)
  [C] : Kéo giãn độ tương phản (Contrast Stretching)
  [A] : Cân bằng lược đồ mức xám thích ứng (Adaptive Histogram Equalization - CLAHE)
  [Q] : Thoát chương trình
  *** HƯỚNG DẪN: Nhấp chuột vào MỘT TRONG CÁC CỬA SỔ ẢNH (Ảnh Gốc 1, 2, 3...)
                 sau đó nhấn phím tương ứng trên bàn phím để áp dụng biến đổi.
                 Các cửa sổ ảnh biến đổi sẽ hiện ra.
                 Nhấn 'Q' để đóng tất cả và thoát.

Bạn đã chọn: Biến đổi ảnh nghịch đảo (Phím I)
Bắt đầu áp dụng biến đổi 'inverse' cho tất cả các ảnh...
   -> Đ

In [ ]:
#Cau 3
import cv2
import numpy as np
import os
import random # <--- Dòng này đã được thêm vào

def process_images_advanced():
    """
    Thực hiện các phép biến đổi và tiền xử lý nâng cao cho ba ảnh được chỉ định.
    """

    # Danh sách các ảnh đầu vào
    image_files = {
        "fruits": "colorful-ripe-tropical-fruits.jpg",
        "quang_ninh": "quang_ninh.jpg",
        "pagoda": "pagoda.jpg"
    }

    # Kiểm tra sự tồn tại của các file ảnh
    for key, filename in image_files.items():
        if not os.path.exists(filename):
            print(f"Lỗi: Không tìm thấy file '{filename}'. Vui lòng đảm bảo file này nằm cùng thư mục với script.")
            return

    # --- Tăng kích thước ảnh colorful-ripe-tropical-fruits.jpg thêm 30 pixel ở cả chiều rộng và chiều cao (0.5 Điểm) ---
    print("\n--- 1. Tăng kích thước ảnh 'colorful-ripe-tropical-fruits.jpg' ---")
    fruits_img = cv2.imread(image_files["fruits"])
    if fruits_img is None:
        print(f"Lỗi: Không thể đọc ảnh '{image_files['fruits']}'.")
    else:
        original_height, original_width = fruits_img.shape[:2]
        new_width = original_width + 30
        new_height = original_height + 30
        
        # Sử dụng interpolation mặc định (INTER_LINEAR) hoặc INTER_CUBIC cho chất lượng tốt hơn
        resized_fruits_img = cv2.resize(fruits_img, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
        
        output_filename = "output_fruits_resized.jpg"
        cv2.imwrite(output_filename, resized_fruits_img)
        print(f"  -> Đã tăng kích thước ảnh '{image_files['fruits']}' lên {new_width}x{new_height}.")
        print(f"  -> Kết quả được lưu tại '{output_filename}'.")
        # cv2.imshow("Fruits Resized", resized_fruits_img)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()


    # --- Xoay ảnh quang-ninh.jpg 45 độ theo chiều kim đồng hồ và lật ngang. (0.5 Điểm) ---
    print("\n--- 2. Xoay và lật ảnh 'quang_ninh.jpg' ---")
    quang_ninh_img = cv2.imread(image_files["quang_ninh"])
    if quang_ninh_img is None:
        print(f"Lỗi: Không thể đọc ảnh '{image_files['quang_ninh']}'.")
    else:
        (h, w) = quang_ninh_img.shape[:2]
        center = (w // 2, h // 2)
        
        # Xoay 45 độ theo chiều kim đồng hồ
        M_rotate = cv2.getRotationMatrix2D(center, -45, 1.0) # -45 độ cho chiều kim đồng hồ
        rotated_quang_ninh_img = cv2.warpAffine(quang_ninh_img, M_rotate, (w, h))
        
        # Lật ngang (Horizontal flip)
        flipped_quang_ninh_img = cv2.flip(rotated_quang_ninh_img, 1) # 1 để lật ngang
        
        output_filename = "output_quang_ninh_rotated_flipped.jpg"
        cv2.imwrite(output_filename, flipped_quang_ninh_img)
        print(f"  -> Đã xoay ảnh '{image_files['quang_ninh']}' 45 độ và lật ngang.")
        print(f"  -> Kết quả được lưu tại '{output_filename}'.")
        # cv2.imshow("Quang Ninh Rotated & Flipped", flipped_quang_ninh_img)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()

    # --- Tăng kích thước ảnh pagoda.jpg lên 5 lần và áp dụng Gaussian Blur với kernel 7x7 để làm mịn. (0.5 Điểm) ---
    print("\n--- 3. Tăng kích thước và làm mịn ảnh 'pagoda.jpg' ---")
    pagoda_img = cv2.imread(image_files["pagoda"])
    if pagoda_img is None:
        print(f"Lỗi: Không thể đọc ảnh '{image_files['pagoda']}'.")
    else:
        # Tăng kích thước lên 5 lần
        magnification_factor = 5
        magnified_pagoda_img = cv2.resize(pagoda_img, None, fx=magnification_factor, fy=magnification_factor, interpolation=cv2.INTER_LINEAR)
        
        # Áp dụng Gaussian Blur với kernel 7x7
        blurred_pagoda_img = cv2.GaussianBlur(magnified_pagoda_img, (7, 7), 0)
        
        output_filename = "output_pagoda_magnified_blurred.jpg"
        cv2.imwrite(output_filename, blurred_pagoda_img)
        print(f"  -> Đã tăng kích thước ảnh '{image_files['pagoda']}' lên 5 lần và áp dụng Gaussian Blur (kernel 7x7).")
        print(f"  -> Kết quả được lưu tại '{output_filename}'.")
        # cv2.imshow("Pagoda Magnified & Blurred", blurred_pagoda_img)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()

    # --- Ứng dụng công thức biến đổi độ sáng và tương phản cho ảnh pagoda.jpg (1.5 Điểm) ---
    # Công thức: Iout(x,y) = α * Iin(x,y) + β
    # α: hệ số tương phản (0.5 - 2.0)
    # β: độ lệch sáng (-50 - 50)
    # Giá trị đầu ra Iout(x,y) phải được giới hạn trong khoảng [0, 255]
    print("\n--- 4. Điều chỉnh độ sáng và tương phản ảnh 'pagoda.jpg' ---")
    if pagoda_img is None: # Kiểm tra lại nếu ảnh chưa được đọc ở phần trước
        pagoda_img = cv2.imread(image_files["pagoda"])
        if pagoda_img is None:
            print(f"Lỗi: Không thể đọc ảnh '{image_files['pagoda']}' cho việc điều chỉnh độ sáng/tương phản.")
            return

    # Chuyển ảnh sang kiểu float để thực hiện phép toán
    adjusted_pagoda_img = pagoda_img.astype(np.float32)

    # Tạo các giá trị ngẫu nhiên cho alpha và beta
    alpha = random.uniform(0.5, 2.0)
    beta = random.uniform(-50, 50)

    print(f"  -> Đang áp dụng điều chỉnh với alpha (tương phản) = {alpha:.2f}, beta (độ sáng) = {beta:.2f}...")

    # Áp dụng công thức và giới hạn giá trị trong khoảng [0, 255]
    adjusted_pagoda_img = alpha * adjusted_pagoda_img + beta
    adjusted_pagoda_img = np.clip(adjusted_pagoda_img, 0, 255).astype(np.uint8)

    output_filename = "output_pagoda_contrast_brightness.jpg"
    cv2.imwrite(output_filename, adjusted_pagoda_img)
    print(f"  -> Đã điều chỉnh độ sáng và tương phản ảnh '{image_files['pagoda']}'.")
    print(f"  -> Kết quả được lưu tại '{output_filename}'.")
    # cv2.imshow("Pagoda Contrast & Brightness Adjusted", adjusted_pagoda_img)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    print("\n--- Tất cả các tác vụ đã hoàn thành! ---")
    print("Vui lòng kiểm tra các file ảnh đầu ra trong cùng thư mục với script.")

if __name__ == "__main__":
    process_images_advanced()


--- 1. Tăng kích thước ảnh 'colorful-ripe-tropical-fruits.jpg' ---
  -> Đã tăng kích thước ảnh 'colorful-ripe-tropical-fruits.jpg' lên 2149x1444.
  -> Kết quả được lưu tại 'output_fruits_resized.jpg'.

--- 2. Xoay và lật ảnh 'quang_ninh.jpg' ---
  -> Đã xoay ảnh 'quang_ninh.jpg' 45 độ và lật ngang.
  -> Kết quả được lưu tại 'output_quang_ninh_rotated_flipped.jpg'.

--- 3. Tăng kích thước và làm mịn ảnh 'pagoda.jpg' ---
  -> Đã tăng kích thước ảnh 'pagoda.jpg' lên 5 lần và áp dụng Gaussian Blur (kernel 7x7).
  -> Kết quả được lưu tại 'output_pagoda_magnified_blurred.jpg'.

--- 4. Điều chỉnh độ sáng và tương phản ảnh 'pagoda.jpg' ---
  -> Đang áp dụng điều chỉnh với alpha (tương phản) = 0.82, beta (độ sáng) = 40.57...
  -> Đã điều chỉnh độ sáng và tương phản ảnh 'pagoda.jpg'.
  -> Kết quả được lưu tại 'output_pagoda_contrast_brightness.jpg'.

--- Tất cả các tác vụ đã hoàn thành! ---
Vui lòng kiểm tra các file ảnh đầu ra trong cùng thư mục với script.
